In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding

In [19]:
reviews = ['Nice food', 'amazing restaurant', 'too good', 'horrible service', 'poor quality', 'needs some improvement']
sentiment = np.array([1,1,1,0,0,0])

In [20]:
one_hot_vectors = one_hot(reviews[0], 30)    #gives the random number for the words in limit of 30

print(one_hot_vectors)

[21, 15]


In [30]:
vocab_size = 50
encoded_reviews = [one_hot(d, vocab_size) for d in reviews]
print(encoded_reviews)

[[33, 24], [11, 4], [42, 28], [30, 36], [1, 38], [34, 32, 38]]


In [31]:
# we need to do padding to make the length equal

max_length =3
padded_reviews = pad_sequences(encoded_reviews, maxlen=max_length, padding='post')
print(padded_reviews)

[[33 24  0]
 [11  4  0]
 [42 28  0]
 [30 36  0]
 [ 1 38  0]
 [34 32 38]]


In [32]:
# let's say embedded vector - 5

embedded_vec = 4

model = Sequential()
model.add(Embedding(vocab_size, embedded_vec, input_length=max_length, name='embedding'))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
X=padded_reviews
y=sentiment

In [34]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.fit(X, y, epochs=50, verbose=0)
print('Model training complete.')

Model training complete.


Now that the model is trained, let's see how it predicts the sentiment for our `padded_reviews`.

In [37]:
y_pred = model.predict(X)
print(y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
[[0.5913562 ]
 [0.5979562 ]
 [0.6033602 ]
 [0.43701082]
 [0.41809344]
 [0.35360327]]


In [38]:
loss, accuracy = model.evaluate(X,y)
accuracy

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 515ms/step - accuracy: 1.0000 - loss: 0.5162


1.0

In [42]:
weights = model.get_layer('embedding').get_weights()[0]

In [43]:
weights[11]

array([-0.10234231, -0.12174567,  0.14426905,  0.07915905], dtype=float32)

In [44]:
weights[3]

array([-0.01158572,  0.01408196,  0.04114847,  0.02550361], dtype=float32)

nice and amazing vectors should be similar but our data is very less to process it

- Embeddings are not handcrafted. Instead they are learnt during neural network training

meaning of the word can be inferred by the surrounding word

--- Word2Vec----

CBOW - continuous bag of words - given context words predict target word
SkipGram - given the target predict context word


In [46]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.6 MB/s eta 0:00:00


In [47]:
import gensim
import pandas as pd

In [55]:
# mount the drive first to read the json file
import google.colab.drive as drive
drive.mount('/content/drive')
df = pd.read_json('/content/drive/MyDrive/Colab Notebooks/Cell_Phones_and_Accessories_5.json', lines=True)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A30TL5EWN6DFXT,120401325X,christina,"[0, 0]",They look good and stick good! I just don't li...,4,Looks Good,1400630400,"05 21, 2014"
1,ASY55RVNIL0UD,120401325X,emily l.,"[0, 0]",These stickers work like the review says they ...,5,Really great product.,1389657600,"01 14, 2014"
2,A2TMXE2AFO7ONB,120401325X,Erica,"[0, 0]",These are awesome and make my phone look so st...,5,LOVE LOVE LOVE,1403740800,"06 26, 2014"
3,AWJ0WZQYMYFQ4,120401325X,JM,"[4, 4]",Item arrived in great time and was in perfect ...,4,Cute!,1382313600,"10 21, 2013"
4,ATX7CZYFXI1KW,120401325X,patrice m rogoza,"[2, 3]","awesome! stays on, and looks great. can be use...",5,leopard home button sticker for iphone 4s,1359849600,"02 3, 2013"


In [56]:
df.shape

(194439, 9)

In [58]:
df.reviewText

,reviewText
0,They look good and stick good! I just don't li...
1,These stickers work like the review says they ...
2,These are awesome and make my phone look so st...
3,Item arrived in great time and was in perfect ...
4,"awesome! stays on, and looks great. can be use..."
...,...
194434,Works great just like my original one. I reall...
194435,Great product. Great packaging. High quality a...
194436,"This is a great cable, just as good as the mor..."
194437,I really like it becasue it works well with my...


In [59]:
df.reviewText[0]

"They look good and stick good! I just don't like the rounded shape because I was always bumping it and Siri kept popping up and it was irritating. I just won't buy a product like this again"

In [61]:
# to remove the spaces, stop words, punctuation marks, lower case

gensim.utils.simple_preprocess("They look good and stick good! I just don't like the rounded shape because I was always bumping it and Siri kept popping up and it was irritating. I just won't buy a product like this again")

['they',
 'look',
 'good',
 'and',
 'stick',
 'good',
 'just',
 'don',
 'like',
 'the',
 'rounded',
 'shape',
 'because',
 'was',
 'always',
 'bumping',
 'it',
 'and',
 'siri',
 'kept',
 'popping',
 'up',
 'and',
 'it',
 'was',
 'irritating',
 'just',
 'won',
 'buy',
 'product',
 'like',
 'this',
 'again']

In [62]:
review_text = df.reviewText.apply(gensim.utils.simple_preprocess)

In [63]:
review_text

,reviewText
0,"[they, look, good, and, stick, good, just, don..."
1,"[these, stickers, work, like, the, review, say..."
2,"[these, are, awesome, and, make, my, phone, lo..."
3,"[item, arrived, in, great, time, and, was, in,..."
4,"[awesome, stays, on, and, looks, great, can, b..."
...,...
194434,"[works, great, just, like, my, original, one, ..."
194435,"[great, product, great, packaging, high, quali..."
194436,"[this, is, great, cable, just, as, good, as, t..."
194437,"[really, like, it, becasue, it, works, well, w..."


In [64]:
model = gensim.models.Word2Vec(
    window=10,      #10 words before and after the target word
    min_count  = 2, #atleast 2 words need to be there to be considered for the training
    workers = 4    #cpu threads to work on this
)

In [65]:
# building vocabulary means building unique list of words
model.build_vocab(review_text, progress_per=1000)

In [66]:
model.epochs

5

In [67]:
# perform the actual training
model.train(review_text, total_examples=model.corpus_count, epochs=model.epochs)


(61509822, 83868975)

In [ ]:
# save the model
model.save('/content/drive/MyDrive/Colab Notebooks/word2vec-amazon-cell-accessories-reviews-short.model')

In [69]:
# experiment
model.wv.most_similar('amazing')

[('awesome', 0.9139083623886108),
 ('fantastic', 0.8113397359848022),
 ('outstanding', 0.769055962562561),
 ('unbelievable', 0.7442076206207275),
 ('wonderful', 0.7344396710395813),
 ('excellent', 0.7179267406463623),
 ('superb', 0.7162193655967712),
 ('fabulous', 0.7133982181549072),
 ('incredible', 0.6878402233123779),
 ('stunning', 0.6788250803947449)]

In [70]:
# comparison b/w words
model.wv.similarity(w1='great', w2='good')

np.float32(0.78280485)

In [71]:
model.wv.similarity('cheap', 'expensive')

np.float32(0.45204875)